# RLHF: 인간 피드백 기반 강화학습 - 실습 코드 2: RLHF 전체 파이프라인 구현 (SFT → RM → PPO)

- Tutorial ID: `expand-rlhf`
- Tutorial: RLHF: 인간 피드백 기반 강화학습
- Section ID: `expand-rlhf-code-2`
- Section: 실습 코드 2: RLHF 전체 파이프라인 구현 (SFT → RM → PPO)


In [1]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: RLHF 전체 파이프라인 구현 (SFT → RM → PPO)
#
# 이 코드는 "정답을 한 번 실행"하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#   2) SFT -> RM -> PPO 각 단계의 손실 함수가 실제 숫자로 어떻게 계산되는지 확인
#   3) PPO의 ratio / clipping / KL penalty가 코드에서 어떻게 구현되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, hidden_dim 등)를 먼저 확인합니다.
#   2) 입력 배열 또는 토큰 데이터가 어떻게 만들어지는지 봅니다.
#   3) 가중치들이 어떤 공간으로 투영하는지 확인합니다 (여기서는 embedding / LSTM / head).
#   4) matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, epoch, lr, epsilon(클리핑 폭) 등을 바꿔가며 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape 변화"와 "정보가 이동하는 방향"을 보세요.
#   - 이 노트북은 torch만으로 동작하는 아주 작은 장난감(toy) 모델을 직접 구현해서 사용하므로,
#     별도의 GPU나 transformers/openai 같은 외부 라이브러리 없이 로컬 CPU에서도
#     처음부터 끝까지 바로 실행해 볼 수 있습니다. (반드시 위에서 아래로 순서대로 실행하세요)
# ============================================================


## 이 실습에서 배우는 것

RLHF는 크게 3단계로 이루어집니다. 신입사원을 교육하는 과정에 비유하면 이렇습니다.

1. **SFT (지도학습 미세조정)** — 신입사원에게 "모범 답안" 예시를 여러 개 보여주고 그대로 따라 하게 시키는 단계입니다. "이런 질문에는 이렇게 답하는구나"를 배웁니다.
2. **RM (보상 모델)** — 이번엔 두 개의 답안을 나란히 보여주고 "어느 쪽이 더 나은 답인지" 채점하는 법을 가르칩니다. 사람이 어느 쪽을 더 선호하는지 알려주고, 그걸 보고 점수를 매기는 채점자(보상 모델)를 학습시킵니다.
3. **PPO (강화학습 최적화)** — 신입사원이 이제 스스로 답을 써 보고, 앞서 만든 채점자에게 점수를 받으면서 점수가 높아지는 방향으로 계속 연습합니다. 단, 채점자를 속이는 편법(이상한 문장인데 점수만 높게 받는 것)을 막기 위해, 원래 배웠던 방식에서 너무 벗어나지 않도록 견제하는 장치도 함께 둡니다.

이 노트북에서는 이 3단계를 각각 다음 순서로 다룹니다.

    왜 필요한지  →  수식이 뭔지  →  작은 숫자로 손으로 계산해보기  →  실제 코드로 실행해보기

실습 코드 1에서 이미 트랜스포머/어텐션 구조를 다뤘다면, 이 노트북은 모델 내부 구조보다는 SFT → RM → PPO로 이어지는 **학습 파이프라인 자체**에 집중합니다.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy      # 모델을 통째로 복제할 때 사용 (예: SFT 모델을 복제해서 ref_model, reward model의 base로 재사용)
import random     # 데이터 셔플, 프롬프트 무작위 선택 등에 사용

# 실행할 때마다 결과가 달라지면 학습/디버깅이 어려우므로 seed를 고정합니다.
random.seed(42)
torch.manual_seed(42)

print("torch version:", torch.__version__)


torch version: 2.12.1+cu130


## 실습 준비: 왜 작은 '장난감(toy)' 모델을 쓰나요?

실제 ChatGPT류 모델은 파라미터가 수십억~수천억 개이고, 학습에는 GPU 클러스터와 며칠~몇 주가 걸립니다. 이 노트북에서 그런 모델을 그대로 쓸 수는 없겠죠.

그래서 아래처럼 모든 것을 아주 작게 축소한 '장난감' 버전을 직접 만들어서 사용합니다.

- **토크나이저**: 실제로는 BPE(Byte Pair Encoding) 같은 복잡한 서브워드 토크나이저를 쓰지만, 여기서는 "띄어쓰기 = 토큰 1개"인 단순한 토크나이저를 씁니다.
- **언어모델**: 실제 GPT는 Self-Attention을 여러 층 쌓지만, 여기서는 구조가 단순하고 결과를 이해하기 쉬운 LSTM(순환 신경망) 기반의 아주 작은 모델을 씁니다. "이전 단어들을 순서대로 읽으며 문맥을 압축하고, 그 문맥으로 다음 단어를 예측한다"는 역할 자체는 GPT와 동일합니다.
- **데이터셋**: 문장 5개짜리 초미니 데이터셋을 씁니다.

이렇게 하면 CPU 노트북에서도 몇 초 안에 SFT → RM → PPO 전체 파이프라인이 실제로 돌아가는 것을 볼 수 있고, 각 단계에서 숫자와 shape이 어떻게 바뀌는지 직접 확인할 수 있습니다. **원리는 실제 대규모 모델과 완전히 동일하며, 다른 점은 오직 "크기"뿐입니다.**


In [3]:
class ToyTokenizer:
    """
    실제 GPT 계열 모델은 BPE 같은 복잡한 토크나이저로 '단어의 일부(subword)'를 토큰으로 쪼갭니다.
    이 실습에서는 원리를 쉽게 보기 위해 "띄어쓰기 단위 = 토큰 1개"인 아주 단순한 토크나이저를 씁니다.
    예) "안녕하세요 반가워요" -> ["안녕하세요", "반가워요"] -> [3, 7] 같은 식으로 정수 id로 바꿔줍니다.
    """
    def __init__(self, vocab):
        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}
        self.vocab_size = len(vocab)
        self.pad_id = self.word2id["<pad>"]   # 빈 자리를 채우는 토큰 (이 노트북에서는 배치를 안 묶어서 거의 안 씀)
        self.unk_id = self.word2id["<unk>"]   # 사전에 없는 단어 대신 쓰는 토큰
        self.bos_id = self.word2id["<bos>"]   # 문장 시작 토큰 (이 노트북에서는 별도로 안 씀, 자리만 마련)
        self.eos_id = self.word2id["<eos>"]   # 문장이 끝났다는 토큰 -> generate()가 멈추는 신호로 사용

    def encode(self, text, return_tensors=None):
        """문자열을 정수 id의 리스트(또는 텐서)로 바꿉니다."""
        ids = [self.word2id.get(w, self.unk_id) for w in text.split()]
        if return_tensors == "pt":
            return torch.tensor([ids], dtype=torch.long)   # (1, seq_len) - 배치 차원 1개를 추가
        return ids

    def decode(self, ids):
        """정수 id의 리스트(또는 텐서)를 다시 문자열로 되돌립니다."""
        if torch.is_tensor(ids):
            ids = ids.tolist()
        words = [self.id2word[i] for i in ids if i != self.pad_id]
        return " ".join(words)


# 이 실습에서 사용할 전체 단어 사전(vocab)입니다.
# 아래 SFT/RM 데이터에 등장하는 모든 단어 + 몇 가지 특수 토큰으로 구성했습니다.
VOCAB = [
    "<pad>", "<unk>", "<bos>", "<eos>",
    "User:", "Assistant:",
    "안녕", "안녕하세요", "무엇을", "도와드릴까요",
    "고마워", "천만에요", "언제든", "물어보세요",
    "2", "더하기", "3은", "뭐야", "5입니다", "10입니다",
    "파이썬이", "파이썬은", "프로그래밍", "언어입니다",
    "잘가", "안녕히", "가세요", "또", "만나요",
    "몰라", "귀찮게", "하지", "마세요", "그냥", "아무거나요",
]

tokenizer = ToyTokenizer(VOCAB)

# 인코딩 -> 디코딩이 잘 맞물리는지 확인해봅니다. (읽는 법 2번째 원칙: 데이터가 어떻게 만들어지는지 보기)
sample = "User: 안녕\nAssistant: 안녕하세요 무엇을 도와드릴까요"
ids = tokenizer.encode(sample, return_tensors="pt")
print("원문        :", sample.replace("\n", " \\n "))
print("토큰 id     :", ids)
print("shape       :", ids.shape, " (배치=1, 시퀀스길이=%d)" % ids.shape[1])
print("복원(decode):", tokenizer.decode(ids[0]))
print("전체 vocab 크기:", tokenizer.vocab_size, "개 단어")


원문        : User: 안녕 \n Assistant: 안녕하세요 무엇을 도와드릴까요
토큰 id     : tensor([[4, 6, 5, 7, 8, 9]])
shape       : torch.Size([1, 6])  (배치=1, 시퀀스길이=6)
복원(decode): User: 안녕 Assistant: 안녕하세요 무엇을 도와드릴까요
전체 vocab 크기: 35 개 단어


## 모델 클래스 만들기

아래 `TinyLM`은 실제 Hugging Face 모델을 `model(input_ids, labels=labels)` 처럼 호출하고 `.logits`, `.loss`, `.last_hidden_state`를 반환받는 방식과 최대한 비슷한 '인터페이스'를 갖도록 만들었습니다. 그래야 나중에 진짜 GPT-2 같은 모델로 바꿔 끼우더라도, 아래에서 만들 SFT / RM / PPO 코드를 거의 그대로 재사용할 수 있습니다.


In [4]:
class LMOutput:
    """
    실제 Hugging Face 모델을 예를 들어 model(input_ids, labels=labels) 처럼 호출하면,
    결과로 .logits / .loss / .last_hidden_state 등을 담은 객체가 돌아옵니다.
    TinyLM도 동일한 '모양'의 결과를 반환하도록 이 클래스로 감싸줍니다.
    """
    def __init__(self, logits, loss=None, last_hidden_state=None):
        self.logits = logits                        # (batch, seq_len, vocab_size) - 각 위치에서 "다음 단어" 점수
        self.loss = loss                             # 스칼라 값 1개 - labels를 줬을 때만 계산됨
        self.last_hidden_state = last_hidden_state    # (batch, seq_len, hidden_dim) - RM이 재활용할 문맥 벡터


class TinyLM(nn.Module):
    """
    아주 작은 '다음 단어 예측' 언어모델입니다.

    실제 GPT는 Self-Attention을 여러 층 쌓아서 문맥을 이해하지만(다른 실습에서 다룹니다),
    이 노트북의 목적은 RLHF '파이프라인'을 이해하는 것이므로,
    내부 구조는 구현이 훨씬 간단한 LSTM(순환 신경망)을 사용합니다.

    동작 방식(직관):
      1) 문장을 한 단어씩 순서대로 읽습니다.
      2) 읽을 때마다 "지금까지 읽은 내용의 요약"인 은닉 상태(hidden state)를 갱신합니다.
      3) 각 시점의 은닉 상태로부터 "다음에 올 단어"의 점수(logits)를 계산합니다.

    파라미터:
      vocab_size : 전체 단어 집합의 크기
      embed_dim  : 단어 1개를 표현하는 벡터의 차원
      hidden_dim : 문맥을 압축해서 담아두는 은닉 상태의 차원
    """
    def __init__(self, vocab_size, embed_dim=24, hidden_dim=48):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)               # 단어 id -> 벡터
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)   # 순서대로 읽으며 문맥을 누적
        self.ln = nn.LayerNorm(hidden_dim)                             # 학습 안정화용 정규화
        self.head = nn.Linear(hidden_dim, vocab_size)                  # 은닉상태 -> 단어별 점수(logits)
        self.hidden_dim = hidden_dim

    def forward(self, input_ids, labels=None):
        # input_ids: (batch, seq_len)  예) [[4, 6, 5, 7, 8, 9]]
        x = self.embed(input_ids)          # (batch, seq_len, embed_dim)
        hidden, _ = self.lstm(x)           # (batch, seq_len, hidden_dim)  <- 각 위치까지의 문맥이 담김
        hidden = self.ln(hidden)
        logits = self.head(hidden)         # (batch, seq_len, vocab_size)

        loss = None
        if labels is not None:
            # "다음 단어 맞히기" 문제이므로 한 칸씩 밀어서 비교합니다.
            # 입력이 [A, B, C, D] 라면
            #   logits[0] (A를 읽은 뒤의 예측)은 B를 맞혀야 하고
            #   logits[1] (A,B를 읽은 뒤의 예측)은 C를 맞혀야 하고
            #   logits[2] (A,B,C를 읽은 뒤의 예측)은 D를 맞혀야 합니다.
            # 그래서 logits는 마지막 한 칸을 자르고(:-1), labels는 첫 한 칸을 자릅니다(1:).
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1),
                ignore_index=-100,   # labels가 -100인 위치는 loss 계산에서 제외 (SFT의 프롬프트 마스킹에 사용)
            )
        return LMOutput(logits=logits, loss=loss, last_hidden_state=hidden)

    @torch.no_grad()   # 생성(generate)은 학습이 아니라 '사용'이므로 그래디언트가 필요 없습니다.
    def generate(self, input_ids, max_new_tokens=8, do_sample=True, eos_id=None, temperature=1.0):
        """input_ids 뒤에 새 토큰을 한 개씩 이어 붙여가며 문장을 만듭니다."""
        self.eval()
        generated = input_ids.clone()
        for _ in range(max_new_tokens):
            out = self.forward(generated)
            next_token_logits = out.logits[:, -1, :] / temperature   # 가장 마지막 위치 = "다음 토큰"에 대한 예측
            if do_sample:
                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)   # 확률적으로 하나 뽑기 (다양성 O)
            else:
                next_token = next_token_logits.argmax(dim=-1, keepdim=True)  # 가장 확률 높은 것만 (항상 동일)
            generated = torch.cat([generated, next_token], dim=1)
            if eos_id is not None and next_token.item() == eos_id:
                break   # 문장이 끝났다는 <eos> 토큰이 나오면 멈춤
        self.train()
        return generated


In [5]:
sft_model = TinyLM(vocab_size=tokenizer.vocab_size)

# 학습을 시작하기 전에, 모델이 어떤 shape의 결과를 내는지부터 확인합니다.
# (읽는 법 4번째 원칙: 핵심 연산 직후의 shape을 출력으로 검증하기)
test_ids = tokenizer.encode("User: 안녕\nAssistant:", return_tensors="pt")
test_out = sft_model(test_ids)

print("입력 input_ids shape        :", test_ids.shape, " (배치=1, 시퀀스길이=3)")
print("출력 logits shape           :", test_out.logits.shape, " (배치, 시퀀스길이, vocab_size=%d)" % tokenizer.vocab_size)
print("출력 last_hidden_state shape:", test_out.last_hidden_state.shape, " (배치, 시퀀스길이, hidden_dim=%d)" % sft_model.hidden_dim)

print()
print("[학습 전] 무작위 초기화 상태의 모델이 만든 응답 (아직 아무것도 배우지 않은 상태):")
sample_out = sft_model.generate(test_ids, max_new_tokens=6, do_sample=True, eos_id=tokenizer.eos_id)
print(" ->", tokenizer.decode(sample_out[0]))
print(" (당연히 아직은 의미 없는 단어 나열입니다. 이제 SFT로 이 모델을 가르쳐 보겠습니다)")


입력 input_ids shape        : torch.Size([1, 3])  (배치=1, 시퀀스길이=3)
출력 logits shape           : torch.Size([1, 3, 35])  (배치, 시퀀스길이, vocab_size=35)
출력 last_hidden_state shape: torch.Size([1, 3, 48])  (배치, 시퀀스길이, hidden_dim=48)

[학습 전] 무작위 초기화 상태의 모델이 만든 응답 (아직 아무것도 배우지 않은 상태):
 -> User: 안녕 Assistant: 안녕 고마워 안녕 파이썬이 5입니다 가세요
 (당연히 아직은 의미 없는 단어 나열입니다. 이제 SFT로 이 모델을 가르쳐 보겠습니다)


## Step 1 — SFT (Supervised Fine-Tuning, 지도학습 미세조정)

**목표**: 사람이 미리 작성해 둔 "질문 → 모범답안" 쌍을 그대로 따라 하도록 가르치는 것.

사전학습(pretraining)만 마친 언어모델은 "그럴듯한 다음 단어"는 잘 예측하지만, `User: ... / Assistant: ...` 같은 대화 형식으로 유용하게 답하는 법은 모릅니다. SFT는 사람이 작성한 좋은 예시들을 정답으로 놓고 지도학습(supervised learning)시키는 첫 단계입니다. 사실 우리가 이미 알고 있는 일반적인 "다음 단어 예측" 학습과 동일하며, 학습 데이터가 '대화 형식'이라는 점만 다릅니다.

핵심 손실 함수:

    Loss = -log π(y | x)

x는 프롬프트(질문), y는 응답(정답), π는 우리가 학습시키는 모델입니다. 즉 "정답 응답 y가 나올 확률을 최대한 높이자"는 뜻이고, 확률의 log에 마이너스를 붙였으니 이 값을 최소화하는 방향으로 학습합니다. (실제로는 응답의 각 단어(토큰)마다 이 loss를 계산해서 평균 냅니다 — `F.cross_entropy`가 정확히 이 계산을 해 줍니다)

**왜 프롬프트 부분은 마스킹(labels=-100)하나요?**

우리가 배우고 싶은 것은 "이런 질문이 오면 이렇게 답한다"이지, "질문 자체를 그대로 예측하는 능력"이 아닙니다. `User: 안녕`이라는 프롬프트는 이미 주어진 것이지 모델이 "만들어내야" 할 대상이 아니죠. 그래서 프롬프트에 해당하는 위치는 `labels`를 `-100`으로 표시해서 loss 계산에서 제외하고, `Assistant:` 이후의 응답 부분에 대해서만 "이 단어를 맞혔는가"를 채점합니다. (`-100`은 PyTorch의 `cross_entropy`가 "이 위치는 무시하라"고 인식하는 특수 값입니다)


In [6]:
# (prompt, 사람이 작성한 모범 응답) 쌍들입니다.
# 실제로는 사람이 수만~수십만 개를 작성하지만, 여기서는 원리를 보기 위한 초미니 데이터셋입니다.
sft_data = [
    ("안녕", "안녕하세요 무엇을 도와드릴까요"),
    ("고마워", "천만에요 언제든 물어보세요"),
    ("2 더하기 3은 뭐야", "2 더하기 3은 5입니다"),
    ("파이썬이 뭐야", "파이썬은 프로그래밍 언어입니다"),
    ("잘가", "안녕히 가세요 또 만나요"),
]

# 위에서 설명한 "프롬프트 마스킹"이 실제로 어떻게 되는지, 데이터 1개만 미리 손으로 뜯어봅니다.
prompt, response = sft_data[0]

# 1) 프롬프트 + 응답을 하나로 이어붙입니다. (문장이 끝났다는 표시로 <eos>도 붙입니다)
full_text = f"User: {prompt}\nAssistant: {response} <eos>"
full_ids = tokenizer.encode(full_text, return_tensors="pt")

# 2) "Assistant:" 까지만 인코딩해서 몇 토큰인지 셉니다 -> 이 길이만큼 마스킹할 것입니다.
prompt_only = f"User: {prompt}\nAssistant:"
prompt_len = len(tokenizer.encode(prompt_only))

print("전체 텍스트 :", full_text.replace("\n", " \\n "))
print("전체 토큰   :", [tokenizer.id2word[i] for i in full_ids[0].tolist()])
print("prompt 길이 :", prompt_len, "토큰")

# 3) 앞의 prompt_len개 위치를 -100으로 덮어씁니다.
labels = full_ids.clone()
labels[:, :prompt_len] = -100
print()
print("최종 labels        :", labels[0].tolist())
print("labels가 가리키는 것:", ["(무시)" if i == -100 else tokenizer.id2word[i] for i in labels[0].tolist()])
print()
print("-> 앞의 3개 위치(User: 안녕 Assistant:)는 -100이라 loss 계산에서 빠지고,")
print("   '안녕하세요 무엇을 도와드릴까요 <eos>' 부분만 실제로 채점됩니다.")


전체 텍스트 : User: 안녕 \n Assistant: 안녕하세요 무엇을 도와드릴까요 <eos>
전체 토큰   : ['User:', '안녕', 'Assistant:', '안녕하세요', '무엇을', '도와드릴까요', '<eos>']
prompt 길이 : 3 토큰

최종 labels        : [-100, -100, -100, 7, 8, 9, 3]
labels가 가리키는 것: ['(무시)', '(무시)', '(무시)', '안녕하세요', '무엇을', '도와드릴까요', '<eos>']

-> 앞의 3개 위치(User: 안녕 Assistant:)는 -100이라 loss 계산에서 빠지고,
   '안녕하세요 무엇을 도와드릴까요 <eos>' 부분만 실제로 채점됩니다.


In [7]:
def train_sft(model, tokenizer, sft_data, epochs=3, lr=2e-5):
    """
    인간이 작성한 (prompt, response) 쌍으로 지도학습(SFT)을 수행합니다.

    실제 서비스에서는 이미 수억~수천억 파라미터로 사전학습된 모델을 아주 살짝만
    건드리는 것이므로 lr=2e-5처럼 매우 작은 학습률을 사용합니다.
    (반면 이 노트북의 장난감 모델은 무작위 초기화 상태에서 처음부터 학습을 시작하므로,
     실제로 실행하는 다음 셀에서는 이보다 훨씬 큰 lr과 더 많은 epoch을 사용합니다)
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        data = list(sft_data)
        random.shuffle(data)   # 매 epoch마다 순서를 섞어서 모델이 "순서"를 외우지 않도록 합니다.

        for prompt, response in data:
            # 1) "프롬프트 + 응답"을 하나의 시퀀스로 만듭니다.
            full_text = f"User: {prompt}\nAssistant: {response} <eos>"
            input_ids = tokenizer.encode(full_text, return_tensors="pt")

            # 2) 응답 부분만 loss를 계산하도록 프롬프트 위치를 마스킹합니다. (위 예시와 동일한 방식)
            prompt_len = len(tokenizer.encode(f"User: {prompt}\nAssistant:"))
            labels = input_ids.clone()
            labels[:, :prompt_len] = -100

            # 3) 표준적인 학습 4단계: forward -> loss -> backward -> update
            outputs = model(input_ids, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad()          # 이전 스텝의 그래디언트 초기화
            loss.backward()                # 그래디언트 계산 (역전파)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 그래디언트 폭주 방지
            optimizer.step()               # 파라미터 업데이트

            total_loss += loss.item()

        avg_loss = total_loss / len(sft_data)
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"SFT Epoch {epoch+1}/{epochs}: loss={avg_loss:.4f}")

    return model


In [8]:
# 장난감 모델은 사전학습이 안 된 '무작위 초기화' 상태이므로
# 실제 SFT(lr=2e-5)보다 훨씬 큰 학습률과 더 많은 epoch이 필요합니다.
sft_model = train_sft(sft_model, tokenizer, sft_data, epochs=10, lr=3e-3)


SFT Epoch 1/10: loss=3.7699
SFT Epoch 2/10: loss=2.7394
SFT Epoch 4/10: loss=1.6295
SFT Epoch 6/10: loss=0.9249
SFT Epoch 8/10: loss=0.4661
SFT Epoch 10/10: loss=0.2099


In [9]:
print("[SFT 학습 후] greedy(가장 확률 높은 단어만 선택)로 생성한 결과:\n")
for prompt, expected in sft_data:
    input_ids = tokenizer.encode(f"User: {prompt}\nAssistant:", return_tensors="pt")
    out_ids = sft_model.generate(input_ids, max_new_tokens=8, do_sample=False, eos_id=tokenizer.eos_id)
    generated_text = tokenizer.decode(out_ids[0])
    print(f"Q: {prompt}")
    print(f"  기대 응답 : {expected}")
    print(f"  모델 응답 : {generated_text}")
    print()

print("do_sample=True (확률적 샘플링)로 여러 번 뽑아보면, 아직 완벽히 수렴하지 않아 가끔 다른 답도 나옵니다:")
custom_prompt = "2 더하기 3은 뭐야"
input_ids = tokenizer.encode(f"User: {custom_prompt}\nAssistant:", return_tensors="pt")
for i in range(3):
    out_ids = sft_model.generate(input_ids, max_new_tokens=8, do_sample=True, eos_id=tokenizer.eos_id)
    print(f"  시도 {i+1}: {tokenizer.decode(out_ids[0])}")
print()
print("바로 이 '가끔 흔들리는' 부분을 이후 PPO 단계에서 보상을 이용해 더 안정적으로 다듬어볼 것입니다.")


[SFT 학습 후] greedy(가장 확률 높은 단어만 선택)로 생성한 결과:

Q: 안녕
  기대 응답 : 안녕하세요 무엇을 도와드릴까요
  모델 응답 : User: 안녕 Assistant: 안녕하세요 무엇을 도와드릴까요 <eos>

Q: 고마워
  기대 응답 : 천만에요 언제든 물어보세요
  모델 응답 : User: 고마워 Assistant: 천만에요 언제든 물어보세요 <eos>

Q: 2 더하기 3은 뭐야
  기대 응답 : 2 더하기 3은 5입니다
  모델 응답 : User: 2 더하기 3은 뭐야 Assistant: 2 더하기 3은 5입니다 <eos>

Q: 파이썬이 뭐야
  기대 응답 : 파이썬은 프로그래밍 언어입니다
  모델 응답 : User: 파이썬이 뭐야 Assistant: 파이썬은 프로그래밍 언어입니다 <eos>

Q: 잘가
  기대 응답 : 안녕히 가세요 또 만나요
  모델 응답 : User: 잘가 Assistant: 안녕히 가세요 또 만나요 <eos>

do_sample=True (확률적 샘플링)로 여러 번 뽑아보면, 아직 완벽히 수렴하지 않아 가끔 다른 답도 나옵니다:
  시도 1: User: 2 더하기 3은 뭐야 Assistant: 2 더하기 3은 5입니다 <eos>
  시도 2: User: 2 더하기 3은 뭐야 Assistant: 2 더하기 <eos>
  시도 3: User: 2 더하기 3은 뭐야 Assistant: 2 더하기 3은 5입니다 몰라 3은 5입니다 천만에요

바로 이 '가끔 흔들리는' 부분을 이후 PPO 단계에서 보상을 이용해 더 안정적으로 다듬어볼 것입니다.


## Step 2 — Reward Model 학습 (Bradley-Terry 모델)

**목표**: 응답 하나를 보고 "이게 얼마나 좋은 응답인지" 점수(reward)를 매기는 채점 모델을 만드는 것.

SFT만으로는 부족합니다. 사람이 미리 써 둔 모범 답안을 흉내내는 것만으로는 "더 도움이 되는지", "더 정확한지", "더 예의 바른지" 같은 미묘한 선호(preference)까지 배우기 어렵습니다. 그래서 사람에게 "두 응답 중 어느 게 더 나은가요?"라고 직접 비교시킨 데이터를 모으고, 이를 바탕으로 점수를 매기는 보상 모델(Reward Model, RM)을 학습시킵니다.

**Bradley-Terry 모델이란?**

원래는 스포츠 랭킹 등에서 "두 선수 A, B가 붙었을 때 A가 이길 확률"을 각자의 '실력 점수' 차이로 설명하는 통계 모델입니다.

    P(A가 B를 이김) = σ(score_A - score_B)

(σ는 시그모이드 함수로, 어떤 실수든 0~1 사이의 확률 값으로 바꿔줍니다)

RLHF에서는 이 아이디어를 그대로 빌려옵니다.
- "경기" = 사람이 응답 A와 B 중 하나를 더 선호한다고 고르는 것
- "실력 점수" = 보상 모델이 매기는 reward 값

사람이 y_w(chosen, 선호된 응답)를 y_l(rejected, 선호되지 않은 응답)보다 선택했다면, 보상 모델도 r(y_w)가 r(y_l)보다 크게 나오도록 학습시켜야겠죠. 그래서 손실 함수는 다음과 같습니다.

    Loss = -log σ(r(x, y_w) - r(x, y_l))

- r(y_w)가 r(y_l)보다 훨씬 크면 → σ(...) ≈ 1 → log(1) ≈ 0 → loss가 작아짐 (좋은 상태)
- r(y_w)가 r(y_l)보다 오히려 작으면 → σ(...) ≈ 0 → log(0) ≈ -∞ → loss가 커짐 (큰 벌점)

말로 풀면: "사람이 선호한 쪽에 더 높은 점수를 줘야 손실이 줄어드는" 아주 자연스러운 손실 함수입니다.


In [10]:
# 위 공식을 아주 작은 숫자로 직접 계산해보며 감을 잡아봅시다.
r_chosen = torch.tensor(2.0)     # 선호 응답에 (가상으로) 매겨진 점수
r_rejected = torch.tensor(0.5)   # 비선호 응답에 (가상으로) 매겨진 점수

diff = r_chosen - r_rejected
prob_chosen_wins = torch.sigmoid(diff)     # "chosen이 더 낫다"고 볼 확률
loss = -F.logsigmoid(diff)                  # 위 확률의 log에 마이너스 (Bradley-Terry Loss)

print("[정상적인 경우: 선호 응답의 점수가 더 높음]")
print(f"  r_chosen={r_chosen.item()}, r_rejected={r_rejected.item()}, 차이={diff.item()}")
print(f"  sigma(차이) = {prob_chosen_wins.item():.4f}  <- 모델이 'chosen이 더 낫다'고 보는 확률")
print(f"  Loss = {loss.item():.4f}  <- 이미 어느정도 맞히고 있어서 loss가 작음")

print()
print("[반대의 경우: 모델이 거꾸로 비선호 응답에 더 높은 점수를 줬다면]")
r_chosen2 = torch.tensor(0.5)
r_rejected2 = torch.tensor(2.0)
loss2 = -F.logsigmoid(r_chosen2 - r_rejected2)
print(f"  r_chosen={r_chosen2.item()}, r_rejected={r_rejected2.item()}")
print(f"  Loss = {loss2.item():.4f}  <- 완전히 틀렸으므로 훨씬 큰 벌점을 받습니다")


[정상적인 경우: 선호 응답의 점수가 더 높음]
  r_chosen=2.0, r_rejected=0.5, 차이=1.5
  sigma(차이) = 0.8176  <- 모델이 'chosen이 더 낫다'고 보는 확률
  Loss = 0.2014  <- 이미 어느정도 맞히고 있어서 loss가 작음

[반대의 경우: 모델이 거꾸로 비선호 응답에 더 높은 점수를 줬다면]
  r_chosen=0.5, r_rejected=2.0
  Loss = 1.7014  <- 완전히 틀렸으므로 훨씬 큰 벌점을 받습니다


In [11]:
# (prompt, 선호 응답 chosen, 비선호 응답 rejected)
# chosen은 SFT 데이터의 정답과 동일하게 두고, rejected는 "오답이거나 / 성의 없거나 / 무례한" 응답으로 구성했습니다.
rm_data = [
    ("안녕", "안녕하세요 무엇을 도와드릴까요", "몰라 귀찮게 하지 마세요"),
    ("고마워", "천만에요 언제든 물어보세요", "귀찮게 하지 마세요"),
    ("2 더하기 3은 뭐야", "2 더하기 3은 5입니다", "2 더하기 3은 10입니다"),
    ("파이썬이 뭐야", "파이썬은 프로그래밍 언어입니다", "그냥 아무거나요"),
    ("잘가", "안녕히 가세요 또 만나요", "몰라"),
]


class RewardModel(nn.Module):
    """
    응답의 품질을 스칼라 점수(reward) 1개로 평가하는 모델.

    구조:  [SFT를 마친 언어모델(base), 고정됨] -> 마지막 토큰의 은닉상태 -> [작은 reward_head] -> 점수 1개

    base 모델로는 방금 SFT를 마친 모델을 재활용합니다. 문장을 읽고 이해하는 능력은
    이미 갖추고 있으니, 그 위에 "점수 매기기" 기능만 새로 얹는 것입니다.

    여기서는 학습을 단순화하기 위해 base는 얼려두고(freeze) reward_head만 학습시킵니다.
    (실무에서는 base까지 함께 미세조정하는 경우도 흔합니다. 그렇게 바꾸려면
     아래 forward의 `with torch.no_grad()` 블록만 지우면 됩니다)
    """
    def __init__(self, base_model, hidden_dim):
        super().__init__()
        self.base = base_model
        for p in self.base.parameters():
            p.requires_grad = False   # base 모델의 파라미터는 학습하지 않음 (고정)

        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, input_ids):
        with torch.no_grad():
            hidden = self.base(input_ids).last_hidden_state    # (batch, seq_len, hidden_dim)
        last_hidden = hidden[:, -1, :]     # 문장을 끝까지 다 읽은 시점의 은닉상태 = "문장 전체를 요약한 벡터"
        reward = self.reward_head(last_hidden)   # (batch, 1)
        return reward.squeeze(-1)          # (batch,)  각 문장마다 점수 1개


In [12]:
def train_reward_model(rm, tokenizer, rm_data, epochs=100, lr=1e-2):
    """
    Bradley-Terry Loss로 보상 모델을 학습합니다. (자세한 수식 설명은 위 마크다운 셀 참고)

        Loss = -log sigma(r(x, y_w) - r(x, y_l))
    """
    optimizer = torch.optim.AdamW(rm.parameters(), lr=lr)
    rm.train()

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        data = list(rm_data)
        random.shuffle(data)

        for prompt, chosen, rejected in data:
            # RM은 "프롬프트+응답 전체"를 보고 점수를 매기므로, SFT와 같은 형식으로 인코딩합니다.
            chosen_ids = tokenizer.encode(f"User: {prompt}\nAssistant: {chosen} <eos>", return_tensors="pt")
            rejected_ids = tokenizer.encode(f"User: {prompt}\nAssistant: {rejected} <eos>", return_tensors="pt")

            r_chosen = rm(chosen_ids)
            r_rejected = rm(rejected_ids)

            # Bradley-Terry Loss
            loss = -F.logsigmoid(r_chosen - r_rejected).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct += (r_chosen > r_rejected).float().item()   # chosen에 더 높은 점수를 줬으면 "정답"
            total += 1

        acc = correct / total * 100
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"RM Epoch {epoch+1}/{epochs}: loss={total_loss/len(rm_data):.4f}, acc={acc:.1f}%")

    return rm


In [13]:
# base 모델은 SFT를 마친 모델을 복제해서 사용합니다.
# (원본 sft_model은 그대로 두고, 잠시 뒤 PPO에서 "기준 모델(ref_model)"로 다시 재활용할 것입니다)
reward_model = RewardModel(copy.deepcopy(sft_model), hidden_dim=sft_model.hidden_dim)
reward_model = train_reward_model(reward_model, tokenizer, rm_data, epochs=100, lr=1e-2)


RM Epoch 1/100: loss=0.7216, acc=40.0%
RM Epoch 20/100: loss=0.0005, acc=100.0%
RM Epoch 40/100: loss=0.0003, acc=100.0%
RM Epoch 60/100: loss=0.0002, acc=100.0%
RM Epoch 80/100: loss=0.0001, acc=100.0%
RM Epoch 100/100: loss=0.0001, acc=100.0%


In [14]:
print("[RM 학습 후] chosen(선호) vs rejected(비선호) 점수 비교\n")

reward_model.eval()
with torch.no_grad():
    for prompt, chosen, rejected in rm_data:
        c_ids = tokenizer.encode(f"User: {prompt}\nAssistant: {chosen} <eos>", return_tensors="pt")
        r_ids = tokenizer.encode(f"User: {prompt}\nAssistant: {rejected} <eos>", return_tensors="pt")
        rc = reward_model(c_ids).item()
        rr = reward_model(r_ids).item()
        mark = "정답을 구분함" if rc > rr else "아직 헷갈림"
        print(f"Q: {prompt}")
        print(f"  chosen   '{chosen}'   -> reward = {rc:+.3f}")
        print(f"  rejected '{rejected}' -> reward = {rr:+.3f}   [{mark}]")
        print()
reward_model.train()


[RM 학습 후] chosen(선호) vs rejected(비선호) 점수 비교

Q: 안녕
  chosen   '안녕하세요 무엇을 도와드릴까요'   -> reward = +6.078
  rejected '몰라 귀찮게 하지 마세요' -> reward = -5.526   [정답을 구분함]

Q: 고마워
  chosen   '천만에요 언제든 물어보세요'   -> reward = +4.749
  rejected '귀찮게 하지 마세요' -> reward = -3.906   [정답을 구분함]

Q: 2 더하기 3은 뭐야
  chosen   '2 더하기 3은 5입니다'   -> reward = +3.896
  rejected '2 더하기 3은 10입니다' -> reward = -4.519   [정답을 구분함]

Q: 파이썬이 뭐야
  chosen   '파이썬은 프로그래밍 언어입니다'   -> reward = +4.965
  rejected '그냥 아무거나요' -> reward = -4.841   [정답을 구분함]

Q: 잘가
  chosen   '안녕히 가세요 또 만나요'   -> reward = +4.053
  rejected '몰라' -> reward = -7.053   [정답을 구분함]



## Step 3 — PPO (Proximal Policy Optimization)

**목표**: 정책(policy, 최종적으로 배포할 모델)이 앞서 만든 보상 모델에서 높은 점수를 받도록 계속 연습시키되, 원래 배웠던 말투/능력에서 너무 멀어지지 않게 하는 것.

핵심 목적함수:

    max E[ r(x, y) - β · KL(π_θ || π_ref) ]

이 한 줄에 PPO가 하려는 일이 다 들어있습니다. 하나씩 뜯어봅시다.

**① 왜 보상(reward)만 최대화하면 안 되나요? — KL penalty가 필요한 이유**

우리가 만든 보상 모델은 완벽하지 않습니다 (데이터도 5개뿐이었죠!). 만약 정책이 "보상 점수만 높이면 장땡"이라고 학습한다면, 보상 모델의 허점을 파고들어 이상한 문장인데도 점수만 높게 받는 편법을 찾아낼 위험이 있습니다 (이를 **reward hacking**이라고 부릅니다). 예를 들어 특정 단어를 반복하거나 문법이 깨졌는데도, 우연히 보상 모델이 좋아하는 패턴이라 높은 점수를 받는 경우입니다.

이를 막기 위해 "원래 SFT 모델(ref_model, 학습 내내 고정)에서 너무 멀어지지 말라"는 페널티를 함께 줍니다. 이것이 KL(Kullback-Leibler) penalty입니다. 정책이 SFT 모델과 비슷한 확률분포를 유지할수록 페널티가 작고, 크게 벗어날수록 페널티가 커집니다.

**② ratio(확률비)와 clipping — 왜 "조금씩만" 업데이트할까요?**

PPO의 핵심 아이디어는 "한 번에 너무 크게 정책을 바꾸지 말자"입니다. 확률비(ratio)는 "정책이 이 응답을 얼마나 더/덜 내놓게 되었는가"를 나타냅니다.

    ratio = π_new(y|x) / π_old(y|x) = exp(log π_new - log π_old)

- ratio = 1.5 → 이 응답을 낼 확률이 50% 늘어남
- ratio = 0.5 → 이 응답을 낼 확률이 절반으로 줄어듦

이 ratio가 너무 큰 폭으로 움직이면(한 번의 업데이트로 정책이 너무 급격히 바뀌면) 학습이 불안정해지기 쉽습니다. 그래서 PPO는 ratio를 [1-epsilon, 1+epsilon] (보통 epsilon=0.2, 즉 0.8~1.2) 범위로 클리핑(clamp)하고, "클리핑 전/후 값 중 더 작은 쪽(min)"을 최종 목적함수로 사용합니다. 아래 미니 예제로 이게 실제로 어떻게 동작하는지 확인해봅시다.

> 참고: 엄밀한 PPO는 "방금 데이터를 뽑을 때의 정책(old policy)"과 "KL 계산 기준이 되는 정책(reference policy)"을 별도로 관리하고, 하나의 롤아웃(rollout)으로 여러 번 업데이트를 반복합니다. 이 노트북에서는 개념을 단순화하기 위해 고정된 ref_model을 두 역할(오래된 정책의 대역 + KL 기준점) 모두에 사용합니다. ref_model은 학습 내내 고정되어 있으므로, policy_model이 학습되며 점점 멀어질수록 ratio도 자연스럽게 1에서 벗어나기 시작합니다.


In [15]:
beta = 0.1

log_prob_policy = torch.tensor(-2.0)  # 정책이 이 응답에 매긴 평균 로그확률 (0에 가까울수록 자신있게 예측한 것)
log_prob_ref    = torch.tensor(-2.5)  # 기준(SFT) 모델이 같은 응답에 매긴 평균 로그확률

kl_approx = log_prob_policy - log_prob_ref
penalty = beta * kl_approx

print(f"log_prob_policy = {log_prob_policy.item()}")
print(f"log_prob_ref    = {log_prob_ref.item()}")
print(f"KL 근사값(policy - ref) = {kl_approx.item():.3f}")
print(f"KL 페널티(beta * 위 값) = {penalty.item():.3f}")
print()
print("-> 정책이 기준 모델보다 이 응답을 '더 자신있게' 예측할수록(log_prob_policy가 더 클수록)")
print("   KL 근사값이 커지고, 페널티도 커집니다. 즉 기준에서 멀어질수록 손해를 보는 구조입니다.")


log_prob_policy = -2.0
log_prob_ref    = -2.5
KL 근사값(policy - ref) = 0.500
KL 페널티(beta * 위 값) = 0.050

-> 정책이 기준 모델보다 이 응답을 '더 자신있게' 예측할수록(log_prob_policy가 더 클수록)
   KL 근사값이 커지고, 페널티도 커집니다. 즉 기준에서 멀어질수록 손해를 보는 구조입니다.


In [16]:
epsilon = 0.2

print(f"{'ratio':>6} {'advantage':>10} {'clipped_ratio':>14} {'objective':>10}")
for ratio_value, advantage_value in [
    (1.0, 1.0),    # 정책이 거의 안 바뀜 + 좋은 advantage
    (1.5, 1.0),    # 정책이 이 응답 쪽으로 크게(50%) 바뀜 + 좋은 advantage
    (0.5, 1.0),    # 정책이 이 응답을 덜 내놓는 쪽으로 바뀜 + 좋은 advantage인데
    (1.5, -1.0),   # 정책이 크게 바뀜 + 나쁜 advantage(마이너스 보상)
]:
    ratio_t = torch.tensor(ratio_value)
    advantage_t = torch.tensor(advantage_value)
    clipped = torch.clamp(ratio_t, 1 - epsilon, 1 + epsilon)
    obj = torch.min(ratio_t * advantage_t, clipped * advantage_t)
    print(f"{ratio_value:>6} {advantage_value:>10} {clipped.item():>14.2f} {obj.item():>10.2f}")

print()
print("2번째 줄(ratio=1.5, advantage=1.0)을 보면: 원래대로면 1.5*1.0=1.5가 되어야 하는데,")
print("클리핑 때문에 1.2로 '상한'이 걸립니다. -> 아무리 advantage가 좋아도,")
print("한 번에 너무 크게 바뀌는 것에 대한 보상은 여기서 제한합니다.")
print()
print("반면 3번째 줄(ratio=0.5)은 클리핑 없이 원래 값(0.5)이 그대로 쓰이고,")
print("4번째 줄(advantage가 음수)도 클리핑되지 않은 더 나쁜 값(-1.5)이 그대로 쓰입니다.")
print("-> PPO의 클리핑은 '너무 좋아진 것처럼 보이는 상황'에만 상한을 씌우는 비대칭적인 장치입니다.")


 ratio  advantage  clipped_ratio  objective
   1.0        1.0           1.00       1.00
   1.5        1.0           1.20       1.20
   0.5        1.0           0.80       0.50
   1.5       -1.0           1.20      -1.50

2번째 줄(ratio=1.5, advantage=1.0)을 보면: 원래대로면 1.5*1.0=1.5가 되어야 하는데,
클리핑 때문에 1.2로 '상한'이 걸립니다. -> 아무리 advantage가 좋아도,
한 번에 너무 크게 바뀌는 것에 대한 보상은 여기서 제한합니다.

반면 3번째 줄(ratio=0.5)은 클리핑 없이 원래 값(0.5)이 그대로 쓰이고,
4번째 줄(advantage가 음수)도 클리핑되지 않은 더 나쁜 값(-1.5)이 그대로 쓰입니다.
-> PPO의 클리핑은 '너무 좋아진 것처럼 보이는 상황'에만 상한을 씌우는 비대칭적인 장치입니다.


In [17]:
def ppo_train_step(policy_model, ref_model, reward_model, prompt, tokenizer,
                    beta=0.1, epsilon=0.2, max_new_tokens=6):
    """PPO 한 스텝: '응답 생성 -> 채점 -> 손실 계산'까지 한 번 수행합니다.
    (실제 파라미터 업데이트는 이 함수 밖에서 loss.backward() / optimizer.step()으로 수행합니다)
    """
    # 1) rollout: 지금 정책으로 응답을 하나 생성합니다. (아직 학습 아님, 그냥 샘플링)
    prompt_ids = tokenizer.encode(f"User: {prompt}\nAssistant:", return_tensors="pt")
    prompt_len = prompt_ids.shape[1]
    with torch.no_grad():
        response = policy_model.generate(prompt_ids, max_new_tokens=max_new_tokens,
                                          do_sample=True, eos_id=tokenizer.eos_id)

    # 프롬프트 부분은 log-prob 계산에서 제외합니다 (SFT 때 배운 것과 똑같은 마스킹 기법입니다!)
    # -> "정책이 실제로 만들어낸 부분"에 대해서만 log-prob을 계산하기 위함입니다.
    labels = response.clone()
    labels[:, :prompt_len] = -100

    # 2) 보상 계산 (RM은 이 단계에서 학습하지 않으므로 그래디언트가 필요 없습니다)
    with torch.no_grad():
        reward = reward_model(response)

    # 3) 지금 정책(policy_model)의 log-prob (그래디언트 필요! 이 값을 통해 정책이 업데이트됩니다)
    log_prob_policy = -policy_model(response, labels=labels).loss

    # 4) 기준 모델(ref_model, SFT 직후로 고정)의 log-prob (그래디언트 불필요)
    with torch.no_grad():
        log_prob_ref = -ref_model(response, labels=labels).loss

    # 5) KL penalty와 최종 보상(advantage)을 계산합니다.
    kl_penalty = beta * (log_prob_policy - log_prob_ref)
    final_reward = reward - kl_penalty
    # advantage는 '목표로 삼을 고정된 값'으로 취급합니다. (실제 PPO에서도 advantage 자체는
    # 그래디언트를 직접 받지 않고, 아래 ratio 항을 통해서만 정책이 업데이트됩니다)
    advantage = final_reward.detach()

    # 6) 확률비(ratio)와 클리핑된 목적함수
    ratio = torch.exp(log_prob_policy - log_prob_ref)
    clipped_ratio = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
    ppo_loss = -torch.min(ratio * advantage, clipped_ratio * advantage).mean()

    return ppo_loss, final_reward.item(), response


In [18]:
# PPO 단계에서 쓸 3개의 모델을 준비합니다.
#   1) policy_model  : 지금부터 계속 업데이트해 나갈 '진짜 학습 대상'
#   2) ref_model     : SFT 직후 상태로 완전히 고정해 둘 '기준점' (KL penalty 계산용)
#   3) reward_model  : 방금 학습한 보상 모델 (PPO 동안은 채점만 하고, 자신은 학습하지 않음)

policy_model = copy.deepcopy(sft_model)

ref_model = copy.deepcopy(sft_model)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()

for p in reward_model.parameters():
    p.requires_grad = False
reward_model.eval()

policy_optimizer = torch.optim.AdamW(policy_model.parameters(), lr=3e-4)

print("policy_model 학습 가능 파라미터 수:", sum(p.numel() for p in policy_model.parameters() if p.requires_grad))
print("ref_model    학습 가능 파라미터 수:", sum(p.numel() for p in ref_model.parameters() if p.requires_grad), "(고정)")
print("reward_model 학습 가능 파라미터 수:", sum(p.numel() for p in reward_model.parameters() if p.requires_grad), "(고정)")


policy_model 학습 가능 파라미터 수: 16859
ref_model    학습 가능 파라미터 수: 0 (고정)
reward_model 학습 가능 파라미터 수: 0 (고정)


In [19]:
def average_reward(model, tokenizer, reward_model, prompts, n_samples=10):
    """주어진 프롬프트들에 대해 응답을 여러 번 샘플링해서 평균 reward를 구합니다."""
    model.eval()
    total = 0.0
    count = 0
    with torch.no_grad():
        for prompt in prompts:
            input_ids = tokenizer.encode(f"User: {prompt}\nAssistant:", return_tensors="pt")
            for _ in range(n_samples):
                response = model.generate(input_ids, max_new_tokens=6, do_sample=True, eos_id=tokenizer.eos_id)
                total += reward_model(response).item()
                count += 1
    model.train()
    return total / count


prompts_for_ppo = [p for p, _ in sft_data]   # 데모용으로 우리가 가진 프롬프트들을 반복 사용합니다.

before = average_reward(policy_model, tokenizer, reward_model, prompts_for_ppo)
print(f"[PPO 학습 시작 전] 평균 reward: {before:.3f}")


[PPO 학습 시작 전] 평균 reward: 4.046


In [20]:
n_ppo_steps = 60

for step in range(1, n_ppo_steps + 1):
    prompt = random.choice(prompts_for_ppo)

    loss, reward_value, response_ids = ppo_train_step(
        policy_model, ref_model, reward_model, prompt, tokenizer
    )

    policy_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
    policy_optimizer.step()

    if step % 10 == 0 or step == 1:
        text = tokenizer.decode(response_ids[0])
        print(f"[step {step:2d}] prompt='{prompt}' final_reward={reward_value:+.3f} ppo_loss={loss.item():.4f}")
        print(f"          생성 결과: {text}")


[step  1] prompt='잘가' final_reward=+2.967 ppo_loss=-2.9671
          생성 결과: User: 잘가 Assistant: 안녕히 가세요 귀찮게 더하기 5입니다 <eos>
[step 10] prompt='2 더하기 3은 뭐야' final_reward=+3.896 ppo_loss=-3.8967
          생성 결과: User: 2 더하기 3은 뭐야 Assistant: 2 더하기 3은 5입니다 <eos>
[step 20] prompt='파이썬이 뭐야' final_reward=+5.247 ppo_loss=-4.7576
          생성 결과: User: 파이썬이 뭐야 Assistant: 2 더하기 도와드릴까요 <eos>
[step 30] prompt='파이썬이 뭐야' final_reward=+4.963 ppo_loss=-5.0517
          생성 결과: User: 파이썬이 뭐야 Assistant: 파이썬은 프로그래밍 언어입니다 <eos>
[step 40] prompt='잘가' final_reward=+4.598 ppo_loss=-4.3374
          생성 결과: User: 잘가 Assistant: 프로그래밍 언어입니다 <eos>
[step 50] prompt='고마워' final_reward=+4.112 ppo_loss=-4.2191
          생성 결과: User: 고마워 Assistant: 안녕히 가세요 또 만나요 <eos>
[step 60] prompt='파이썬이 뭐야' final_reward=+4.960 ppo_loss=-5.2002
          생성 결과: User: 파이썬이 뭐야 Assistant: 파이썬은 프로그래밍 언어입니다 <eos>


In [21]:
after = average_reward(policy_model, tokenizer, reward_model, prompts_for_ppo)

print(f"[PPO 학습 전] 평균 reward: {before:.3f}")
print(f"[PPO 학습 후] 평균 reward: {after:.3f}")
print(f"변화                    : {after - before:+.3f}")
print()
print("우리 실습은 데이터가 5개뿐이고 모델도 매우 작아서, 매 스텝 깔끔하게 증가하기보다는")
print("진동(oscillation)하면서 전체적으로 조금씩 개선되는 모습을 보입니다.")
print("실제 RLHF에서는 훨씬 많은 프롬프트/스텝, 그리고 value function(baseline)을 활용한")
print("분산 감소 기법(GAE 등)을 함께 사용해서 더 안정적으로 학습합니다.")


[PPO 학습 전] 평균 reward: 4.046
[PPO 학습 후] 평균 reward: 4.408
변화                    : +0.362

우리 실습은 데이터가 5개뿐이고 모델도 매우 작아서, 매 스텝 깔끔하게 증가하기보다는
진동(oscillation)하면서 전체적으로 조금씩 개선되는 모습을 보입니다.
실제 RLHF에서는 훨씬 많은 프롬프트/스텝, 그리고 value function(baseline)을 활용한
분산 감소 기법(GAE 등)을 함께 사용해서 더 안정적으로 학습합니다.


## 정리

지금까지 SFT → RM → PPO로 이어지는 RLHF 전체 파이프라인을 작은 모델로 직접 돌려봤습니다.

- **1. SFT** — 사람이 쓴 모범 답안을 그대로 따라 하기 (Loss = -log π(y|x))
- **2. RM** — 두 응답 중 사람이 선호하는 쪽에 높은 점수 주기 (Loss = -log σ(r(x,y_w) - r(x,y_l)))
- **3. PPO** — 보상은 높이되 원래 모델(ref)에서 너무 벗어나지 않기 (max E[r(x,y) - β·KL(π_θ \|\| π_ref)], 클리핑으로 안정화)

**이 노트북에서 단순화한 부분들** (실제 구현체와의 차이)

- **모델 크기/구조**: 실제로는 수십억 파라미터의 Transformer, 여기서는 파라미터 몇만 개짜리 LSTM
- **토크나이저**: 실제로는 BPE 등 서브워드 토크나이저, 여기서는 띄어쓰기 기반 단순 토크나이저
- **advantage 계산**: 실제로는 별도의 value model(비평가, critic)과 GAE(Generalized Advantage Estimation)로 분산을 줄인 advantage를 쓰는 경우가 많습니다. 여기서는 final_reward를 그대로 advantage로 사용하는 단순화 버전입니다.
- **old policy vs reference model**: 실제로는 "방금 롤아웃할 때의 정책(old)"과 "KL 기준점(ref)"을 구분하고, 하나의 롤아웃 데이터로 여러 번 업데이트(inner epoch)를 반복합니다. 여기서는 ref_model 하나가 두 역할을 겸합니다.
- **데이터 규모**: 실제로는 수만~수십만 개의 사람 작성/비교 데이터, 여기서는 각 5개뿐입니다.

원리 자체(loss 수식, 마스킹 기법, ratio / clipping / KL의 역할)는 실제 프로덕션 RLHF 구현체(TRL, DeepSpeed-Chat 등)와 동일합니다. 이제 실제 라이브러리 코드를 볼 때 "아, 이 부분이 SFT의 마스킹이구나", "여기가 Bradley-Terry loss구나", "여기가 클리핑이구나" 하고 대응시켜볼 수 있을 것입니다.

**더 공부해보고 싶다면**
- DPO(Direct Preference Optimization): RM과 PPO 단계를 하나의 손실 함수로 합친 최신 대안
- GAE(Generalized Advantage Estimation): advantage의 분산을 줄이는 기법
- Constitutional AI / RLAIF: 사람 대신 AI가 선호 데이터를 만드는 방식
